In [230]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc

os.chdir("..")
%run ./pyflowsolver/volumeManager.py
%run ./pyflowsolver/sparseArray.py
%run ./pyflowsolver/fastLaplacian.py
%run ./pyflowsolver/darcySolver.py
os.chdir("notebooks")

In [231]:
import pandas as pd

## minirede
# pores_table = pd.DataFrame({"pore.bc.value": [1, np.nan, np.nan, np.nan, 0]})
# throats_table = pd.DataFrame({
#     "throat.conns_0": [0, 1, 2, 3, 1],
#     "throat.conns_1": [1, 2, 3, 4, 3],
#     "throat.sub_conductivity": [10, 0.2, 2, 5, 1],
# })

## rede real
pores_table=pd.read_pickle("/home/romcenci/Desktop/pore.pd")
throats_table=pd.read_pickle("/home/romcenci/Desktop/throat.pd")

P1 = 101325.0
P2 = 0.0

inlets = []
outlets = []

matrix = np.zeros((len(pores_table),len(pores_table)))
vector = np.zeros(len(pores_table))
for i in range(len(throats_table)):
    conn0 = throats_table["throat.conns_0"][i]
    conn1 = throats_table["throat.conns_1"][i]
    cond = throats_table["throat.sub_conductivity"][i]
    if ~np.isnan(pores_table["pore.bc.value"][conn0]):
        vector[conn1] -= cond*pores_table["pore.bc.value"][conn0]
        matrix[conn1][conn1] -= cond
        if pores_table["pore.bc.value"][conn0] == P2:
            outlets.append((conn0, conn1, i))
        elif pores_table["pore.bc.value"][conn0] == P1:
            inlets.append((conn0, conn1, i))
    elif ~np.isnan(pores_table["pore.bc.value"][conn1]):
        vector[conn0] -= cond*pores_table["pore.bc.value"][conn1]
        matrix[conn0][conn0] -= cond
        if pores_table["pore.bc.value"][conn1] == P2:
            outlets.append((conn0, conn1, i))
        elif pores_table["pore.bc.value"][conn1] == P1:
            inlets.append((conn0, conn1, i))
    else:
        matrix[conn0][conn1] = cond
        matrix[conn1][conn0] = cond

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if i!=j:
            matrix[i][i] -= matrix[i][j]

matrix = matrix[np.isnan(pores_table["pore.bc.value"])][:,np.isnan(pores_table["pore.bc.value"])]
vector = vector[np.isnan(pores_table["pore.bc.value"])]

In [232]:
SIZE = 20
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

volume_manager = VolumeManager(cond_vol)

image OK


In [233]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_from_dense(matrix, vector)

In [234]:
sparse_b = vector

In [248]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations, # sqrt(n) for n x n system
        target_error=1.0e-10, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
# solution

In [249]:
non_nan_indices = np.where(np.isnan(pores_table["pore.bc.value"]))[0]
inverse_map = np.zeros(len(pores_table["pore.bc.value"]))
for i, idx in enumerate(non_nan_indices):
    inverse_map[idx] = i

vazao = 0.0

for outlet in outlets:
    conductivity = throats_table["throat.sub_conductivity"][outlet[2]]
    vazao += (solution[int(inverse_map[outlet[0]])]-P2)*conductivity

for inlet in inlets:
    conductivity = throats_table["throat.sub_conductivity"][inlet[2]]
    vazao -= (solution[int(inverse_map[inlet[0]])]-P1)*conductivity

In [250]:
vazao

0.36033904908601716